# Formularios: uso, lentitud e impacto

Este cuaderno separa tres perspectivas: formularios mas lentos, mas usados y con mayor impacto total en tiempo consumido.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd()
candidate_roots = [
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == 'notebooks' else NOTEBOOK_CWD,
    NOTEBOOK_CWD / 'observability' / 'd365-fo-observability',
]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'kql_runner.py').exists() and (candidate / 'queries').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('No se pudo localizar observability/d365-fo-observability desde el directorio actual')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from kql_runner import build_client, load_config, load_kql, plot_bar, run_kql, set_analyst_theme, summarize_top_items

config = load_config()
client = build_client(config=config)
FORMS_DAYS = config['query_days']
TOP_N = 15
set_analyst_theme()

In [ ]:
df_slow_forms = run_kql(client, load_kql('20_slow_forms.kql'), days=FORMS_DAYS, name='Slow forms', config=config)
display(df_slow_forms.head(TOP_N))
plot_bar(df_slow_forms, 'FormName', 'P95DurationSeconds', 'Formularios con peor P95', top_n=TOP_N)

In [ ]:
df_most_used = run_kql(client, load_kql('21_most_used_forms.kql'), days=FORMS_DAYS, name='Most used forms', config=config)
display(df_most_used.head(TOP_N))
plot_bar(df_most_used, 'FormName', 'FormOpens', 'Formularios mas usados', top_n=TOP_N)

df_total_impact = run_kql(client, load_kql('22_forms_total_impact.kql'), days=FORMS_DAYS, name='Forms total impact', config=config)
display(df_total_impact.head(TOP_N))
display(summarize_top_items(df_total_impact, 'FormName', 'TotalDurationMinutes', top_n=5))
plot_bar(df_total_impact, 'FormName', 'TotalDurationMinutes', 'Formularios con mayor impacto total', top_n=TOP_N)

## Criterios de priorizacion

- P95 alto indica dolor potencial; uso alto indica alcance; impacto total combina ambas cosas.
- Si un formulario es muy lento pero casi no se usa, puede ir a backlog y no al dashboard principal.
- Si buscas una sola visualizacion candidata, TotalDurationMinutes suele ser la mas accionable para negocio y operaciones.